# Camada Gold TS_ALUNO Agregado

**Pipeline:** Análise da Alfabetização no Brasil
**Responsabilidade:** agregar o TS_ALUNO por município/ano, gerando indicadores prontos para cruzar com o `fato_indicador`.

**Decisões de escopo:**
- **Filtro de presença:** só entram alunos com `IN_PRESENCA_LP = 1` — quem faltou não tem proficiência nem alfabetização válida, incluí-los distorceria a média.
- **Granularidade:** agregação por `id_municipio + ano`, mesma chave do `fato_indicador` já existente na Gold principal — permite comparação direta no Power BI.
- **Métricas geradas:** % de alunos alfabetizados, proficiência média e total de alunos avaliados (peso estatístico da agregação).

## 1. Configuração de Acesso

In [1]:
# ============================================================
# Configuração de acesso ao ADLS Gen2 via Azure Key Vault
# ============================================================
from notebookutils import mssparkutils

CONTA_STORAGE = "stalfalfabetizacao"

CHAVE_ACESSO = mssparkutils.credentials.getSecret(
    "https://kv-alfabetizacao.vault.azure.net/",
    "storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{CONTA_STORAGE}.dfs.core.windows.net",
    CHAVE_ACESSO
)

CAMINHO_SILVER = f"abfss://silver@{CONTA_STORAGE}.dfs.core.windows.net/"
CAMINHO_GOLD   = f"abfss://gold@{CONTA_STORAGE}.dfs.core.windows.net/"

print("Conexão configurada com sucesso!")

StatementMeta(sparkpool, 20, 2, Finished, Available, Finished, False)

Conexão configurada com sucesso!


## 2. Leitura e Agregação

Leitura do TS_ALUNO Silver, filtrando apenas alunos presentes na prova. Agregação por `id_municipio + ano`: percentual de alfabetizados, proficiência média e total de alunos

In [2]:
# ============================================================
# Leitura da Silver e agregação por município/ano
# Filtro: apenas alunos presentes na prova (IN_PRESENCA_LP = 1)
# ============================================================
from pyspark.sql.functions import col, avg, count, sum as _sum, round as _round

df_ts_aluno = spark.read.parquet(CAMINHO_SILVER + "ts_aluno/")

df_ts_aluno_presentes = df_ts_aluno.filter(col("IN_PRESENCA_LP") == 1)

df_ts_aluno_agregado = df_ts_aluno_presentes \
    .groupBy(
        col("CO_MUNICIPIO").alias("id_municipio"),
        col("NU_ANO_AVALIACAO").alias("ano")
    ) \
    .agg(
        _round(avg("VL_PROFICIENCIA_LP"), 2).alias("proficiencia_media"),
        _round(avg("IN_ALFABETIZADO") * 100, 2).alias("percentual_alfabetizados"),
        count("ID_ALUNO").alias("total_alunos_avaliados")
    )

print(f"Municípios/anos agregados: {df_ts_aluno_agregado.count()}")
df_ts_aluno_agregado.show(5)

StatementMeta(sparkpool, 20, 3, Finished, Available, Finished, False)

Municípios/anos agregados: 5556
+------------+----+------------------+------------------------+----------------------+
|id_municipio| ano|proficiencia_media|percentual_alfabetizados|total_alunos_avaliados|
+------------+----+------------------+------------------------+----------------------+
|     3540200|2025|            736.06|                   53.95|                   430|
|     1302504|2025|            740.19|                   53.87|                  1368|
|     2312908|2025|             834.6|                   95.88|                  2549|
|     2601003|2025|            762.36|                   71.31|                   122|
|     2601409|2025|            749.49|                    53.9|                   462|
+------------+----+------------------+------------------------+----------------------+
only showing top 5 rows



## 3. Gravação em Parquet

Persistência do agregado na Gold, pronto para cruzar com o `fato_indicador` via `id_municipio + ano`.

In [3]:
# ============================================================
# Gravação do TS_ALUNO agregado em Parquet na Gold
# ============================================================

df_ts_aluno_agregado.write.mode("overwrite").parquet(CAMINHO_GOLD + "fato_aluno_agregado/")

print("TS_ALUNO agregado gravado na Gold:")
print(f"  {CAMINHO_GOLD}fato_aluno_agregado/")

StatementMeta(sparkpool, 20, 4, Finished, Available, Finished, False)

TS_ALUNO agregado gravado na Gold:
  abfss://gold@stalfalfabetizacao.dfs.core.windows.net/fato_aluno_agregado/
